# Firma digital RSA y paquete JSON

Este notebook simula un envío seguro entre Alice y Bob usando RSA, firma digital, cifrado educativo y JSON.

## Librerías

Usamos librerías generales de Python. No usamos librerías criptográficas externas. Para poder cumplir con los requerimentos de la actividad

In [1]:
import base64
import hashlib
import json
import os
import random
import time

## Funciones matemáticas

Estas funciones se usan para generar las llaves RSA.

In [2]:
def mcd(a, b):  # Funcion para calcular el maximo comun divisor entre 2 nums
    while b:
        a, b = b, a % b
    return a


def inverso_modular(e, phi):  # La vimos en clase, calcular el inverso modular
    phi_original = phi
    a, b = e, phi
    x0, x1 = 1, 0

    while b:
        cociente = a // b
        a, b = b, a % b
        x0, x1 = x1, x0 - cociente * x1

    if a != 1:
        raise ValueError("No existe inverso modular")

    return x0 % phi_original


def es_primo(n, rondas=10):  # Funcion por shoots para ver si es primo, usando Miller-Rabbin. Igual que la actividad pasada
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    r = 0
    d = n - 1
    while d % 2 == 0:
        r += 1
        d //= 2

    for _ in range(rondas):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True


def primo_aleatorio(bits): # Generar un numero primo aleatorio, reutiliza la funcion es_primo
    while True:
        candidato = random.getrandbits(bits) | (1 << (bits - 1)) | 1
        if es_primo(candidato):
            return candidato

## Generación de llaves

Cada usuario tiene una llave pública `(n, e)` y una llave privada `(n, d)`.

In [3]:
def generar_llaves(bits=512):  # Se generan llaves (n,e) y (n,d)
    p = primo_aleatorio(bits // 2)
    q = primo_aleatorio(bits // 2)
    while p == q:
        q = primo_aleatorio(bits // 2)

    n = p * q
    phi = (p - 1) * (q - 1)
    e = 65537

    if mcd(e, phi) != 1:
        e = 3
        while mcd(e, phi) != 1:
            e += 2

    d = inverso_modular(e, phi)

    print("p =", p)
    print("q =", q)
    print("n =", n)
    print("phi(n) =", phi)
    print("e =", e)
    print("d =", d)
    print("e*d mod phi(n) =", (e * d) % phi)

    return (n, e), (n, d)


def crear_usuario(nombre, bits=512): # Crea un usuario con su propias llaves RSA
    print("\nCreando usuario:", nombre)
    publica, privada = generar_llaves(bits)
    return {"nombre": nombre, "publica": publica, "privada": privada}


def llave_publica_a_json(llave_publica):
    n, e = llave_publica
    return {"n": str(n), "e": str(e)}


def llave_publica_desde_json(valor):
    if not isinstance(valor, dict) or "n" not in valor or "e" not in valor:
        raise ValueError("La llave pública debe contener n y e")
    return int(valor["n"]), int(valor["e"])

## Mensaje y firma digital

El mensaje se convierte a SHA-256 y ese hash se firma con la llave privada del emisor.

In [4]:
def validar_mensaje(mensaje): # Validamos que el mensaje este correcto para nuestro proceso, que no sean numeros ni este vacio
    if not isinstance(mensaje, str):
        return False, "El mensaje debe ser texto"
    if not mensaje.strip():
        return False, "El mensaje no puede estar vacío"
    return True, "OK"


def hash_mensaje(mensaje):   # hasheamos el mensaje usando sha256
    digest = hashlib.sha256(mensaje.encode("utf-8")).digest()
    return digest.hex(), int.from_bytes(digest, "big")


def firmar(mensaje, llave_privada):  # firma
    ok, razon = validar_mensaje(mensaje)
    if not ok:
        raise ValueError(razon)

    n, d = llave_privada
    hash_hex, h = hash_mensaje(mensaje)

    if h >= n:
        raise ValueError("El hash es mayor o igual que n")

    firma = pow(h, d, n)
    print("Mensaje:", mensaje)
    print("SHA-256:", hash_hex)
    print("Firma:", firma)
    return firma


def verificar(mensaje, firma, llave_publica):   #verificar mensaje usando la llave publica del que manda el mensaje
    ok, razon = validar_mensaje(mensaje)
    if not ok:
        return False, "Mensaje inválido: " + razon

    if not isinstance(firma, int) or firma < 0:
        return False, "La firma debe ser un entero positivo"

    n, e = llave_publica
    if firma >= n:
        return False, "La firma es mayor o igual que n"

    hash_recuperado = pow(firma, e, n)
    hash_hex, hash_real = hash_mensaje(mensaje)

    print("Hash del mensaje:", hash_hex)
    print("Hash recuperado:", hash_recuperado)

    if hash_recuperado == hash_real:
        return True, "Firma válida. El mensaje no fue alterado."
    return False, "Firma inválida. El mensaje o la firma no corresponden."

## Cifrado del mensaje

El mensaje se cifra con una clave de sesión. Luego esa clave se protege con RSA.

In [5]:
def generar_clave_sesion(bits=128):   # Genera una clave de sesion educativa para cifrar el mensaje
    return random.getrandbits(bits) | (1 << (bits - 1))


def flujo_clave(clave_sesion, longitud):  #Flujo de bytes a partir de la clave de sesion^
    salida = bytearray()
    contador = 0

    while len(salida) < longitud:
        semilla = f"{clave_sesion}:{contador}".encode("utf-8")
        salida.extend(hashlib.sha256(semilla).digest())
        contador += 1

    return bytes(salida[:longitud])


def xor_bytes(datos, llave):
    return bytes(dato ^ llave[i] for i, dato in enumerate(datos))


def cifrar_mensaje(mensaje, clave_sesion): 
    ok, razon = validar_mensaje(mensaje)
    if not ok:
        raise ValueError(razon)

    datos = mensaje.encode("utf-8")
    llave = flujo_clave(clave_sesion, len(datos))
    cifrado = xor_bytes(datos, llave)
    return base64.b64encode(cifrado).decode("ascii")


def descifrar_mensaje(mensaje_cifrado, clave_sesion):
    if not isinstance(mensaje_cifrado, str) or not mensaje_cifrado.strip():
        raise ValueError("El mensaje cifrado debe ser texto base64")

    datos_cifrados = base64.b64decode(mensaje_cifrado.encode("ascii"), validate=True)
    llave = flujo_clave(clave_sesion, len(datos_cifrados))
    datos = xor_bytes(datos_cifrados, llave)
    return datos.decode("utf-8")


def cifrar_clave_sesion(clave_sesion, llave_publica_receptor):
    n, e = llave_publica_receptor
    if clave_sesion <= 0 or clave_sesion >= n:
        raise ValueError("La clave de sesión debe ser menor que n")
    return pow(clave_sesion, e, n)


def descifrar_clave_sesion(clave_cifrada, llave_privada_receptor):
    n, d = llave_privada_receptor
    if not isinstance(clave_cifrada, int) or clave_cifrada < 0:
        raise ValueError("La clave cifrada debe ser un entero positivo")
    if clave_cifrada >= n:
        raise ValueError("La clave cifrada es mayor o igual que n")
    return pow(clave_cifrada, d, n)

## Paquete JSON

El emisor crea un paquete con mensaje cifrado, clave cifrada, firma y llave pública.

In [6]:
def crear_paquete(emisor, receptor, mensaje):    # construir paquete de transmision
    ok, razon = validar_mensaje(mensaje)
    if not ok:
        raise ValueError(razon)

    clave_sesion = generar_clave_sesion()
    mensaje_cifrado = cifrar_mensaje(mensaje, clave_sesion)
    clave_sesion_cifrada = cifrar_clave_sesion(clave_sesion, receptor["publica"])
    firma = firmar(mensaje, emisor["privada"])

    paquete = {
        "sender": emisor["nombre"],
        "receiver": receptor["nombre"],
        "encrypted_message": mensaje_cifrado,
        "encrypted_session_key": str(clave_sesion_cifrada),
        "signature": str(firma),
        "hash_algorithm": "SHA-256",
        "sender_public_key": llave_publica_a_json(emisor["publica"])
    }

    print("\nPaquete JSON:")
    print(json.dumps(paquete, indent=2, ensure_ascii=False))
    return paquete


def guardar_paquete_json(paquete, ruta): # guardar en JSON
    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(paquete, archivo, indent=2, ensure_ascii=False)
    print("Archivo guardado:", ruta)


def cargar_paquete_json(ruta):
    with open(ruta, "r", encoding="utf-8") as archivo:
        return json.load(archivo)


def copiar_paquete(paquete):
    return json.loads(json.dumps(paquete))

## Recepción del paquete

Bob descifra la clave de sesión, recupera el mensaje y verifica la firma de Alice.

In [7]:
def validar_estructura_paquete(paquete):
    campos = [
        "sender",
        "receiver",
        "encrypted_message",
        "encrypted_session_key",
        "signature",
        "hash_algorithm",
        "sender_public_key"
    ]
    errores = []

    for campo in campos:
        if campo not in paquete:
            errores.append("Falta el campo: " + campo)

    for campo in ["encrypted_session_key", "signature"]:
        if campo in paquete:
            try:
                int(paquete[campo])
            except ValueError:
                errores.append("El campo " + campo + " debe ser entero")

    if paquete.get("hash_algorithm") != "SHA-256":
        errores.append("El algoritmo hash debe ser SHA-256")

    return errores


def recibir_paquete(paquete, llave_privada_receptor, llave_publica_emisor_esperada=None):
    if isinstance(paquete, str):
        paquete = cargar_paquete_json(paquete)

    print("\nRecibiendo paquete de", paquete.get("sender"), "para", paquete.get("receiver"))

    errores = validar_estructura_paquete(paquete)
    if errores:
        print("Paquete inválido:", errores)
        return False

    try:
        llave_publica_emisor = llave_publica_desde_json(paquete["sender_public_key"])

        if llave_publica_emisor_esperada and llave_publica_emisor != llave_publica_emisor_esperada:
            print("La llave pública no corresponde al emisor esperado")
            return False

        clave_cifrada = int(paquete["encrypted_session_key"])
        firma = int(paquete["signature"])

        clave_sesion = descifrar_clave_sesion(clave_cifrada, llave_privada_receptor)
        mensaje = descifrar_mensaje(paquete["encrypted_message"], clave_sesion)

        print("Mensaje descifrado:", mensaje)

        valido, razon = verificar(mensaje, firma, llave_publica_emisor)
        print("Resultado:", razon)
        return valido

    except Exception as error:
        print("Error al procesar paquete:", error)
        return False

## Pruebas principales

Creamos usuarios, enviamos un mensaje válido y probamos los errores pedidos.

In [8]:
alice = crear_usuario("Alice")
bob = crear_usuario("Bob")
mallory = crear_usuario("Mallory")

mensaje = "Transferencia autorizada: $500 a cuenta 4321"

print("\nCASO 1: mensaje válido")
paquete_valido = crear_paquete(alice, bob, mensaje)
recibir_paquete(paquete_valido, bob["privada"], alice["publica"])

print("\nCASO 2: mensaje vacío")
try:
    crear_paquete(alice, bob, "")
except ValueError as error:
    print("Error capturado:", error)

print("\nCASO 3: mensaje alterado después de firmar")
paquete_alterado = copiar_paquete(paquete_valido)
clave = descifrar_clave_sesion(int(paquete_alterado["encrypted_session_key"]), bob["privada"])
paquete_alterado["encrypted_message"] = cifrar_mensaje("Transferencia autorizada: $900 a cuenta 4321", clave)
recibir_paquete(paquete_alterado, bob["privada"], alice["publica"])

print("\nCASO 4: firma alterada")
paquete_firma_alterada = copiar_paquete(paquete_valido)
paquete_firma_alterada["signature"] = str(int(paquete_firma_alterada["signature"]) + 1)
recibir_paquete(paquete_firma_alterada, bob["privada"], alice["publica"])

print("\nCASO 5: llave pública incorrecta")
paquete_llave_incorrecta = copiar_paquete(paquete_valido)
paquete_llave_incorrecta["sender_public_key"] = llave_publica_a_json(mallory["publica"])
recibir_paquete(paquete_llave_incorrecta, bob["privada"], alice["publica"])

print("\nCASO 6: descifrado con llave privada incorrecta")
recibir_paquete(paquete_valido, mallory["privada"], alice["publica"])

print("\nCASO 7: paquete JSON incompleto")
paquete_incompleto = copiar_paquete(paquete_valido)
del paquete_incompleto["encrypted_message"]
recibir_paquete(paquete_incompleto, bob["privada"], alice["publica"])

print("\nCASO 8: paquete JSON con datos mal formados")
paquete_mal_formado = copiar_paquete(paquete_valido)
paquete_mal_formado["encrypted_session_key"] = "abc"
paquete_mal_formado["signature"] = "xyz"
recibir_paquete(paquete_mal_formado, bob["privada"], alice["publica"])

print("\nCASO 9: firma que no corresponde al mensaje")
paquete_otro = crear_paquete(alice, bob, "Mensaje diferente")
paquete_otro["signature"] = paquete_valido["signature"]
recibir_paquete(paquete_otro, bob["privada"], alice["publica"])


Creando usuario: Alice
p = 112493226402588064439118715857816325145586783875712677045508505304832085237127
q = 89970838971295901461564118176770369429275400652674110899988701718517516703187
n = 10121109958028783273407110473821838458155214694601796699985471682669079724082558495256584650389426802757041016501724514793201519753967703496722538171623749
phi(n) = 10121109958028783273407110473821838458155214694601796699985471682669079724082356031191210766423526119923006429807149652608673132966022206289699188569683436
e = 65537
d = 2166856795720003328031114745230849984075801720859328463272596444444052330875681484865707589051810351231208374149322013149095819592692030096750985165955541
e*d mod phi(n) = 1

Creando usuario: Bob
p = 94145265939038863783420443688360832483306300798751426907092852221541008394427
q = 89097850486078646747258472245827202907125026535895751443934183719279505170691
n = 838814082860859729485440422268576060738668470598391240484765248430432153230570044030577268881091908541861585

False

## Archivos JSON de prueba

Guardamos tres paquetes para entregar y probar.

In [9]:
os.makedirs("json_pruebas", exist_ok=True)

guardar_paquete_json(paquete_valido, "json_pruebas/01_paquete_valido.json")
guardar_paquete_json(paquete_firma_alterada, "json_pruebas/02_paquete_firma_alterada.json")
guardar_paquete_json(paquete_incompleto, "json_pruebas/03_paquete_incompleto.json")

Archivo guardado: json_pruebas/01_paquete_valido.json
Archivo guardado: json_pruebas/02_paquete_firma_alterada.json
Archivo guardado: json_pruebas/03_paquete_incompleto.json
